In [14]:
!pip install pyrosettacolabsetup
import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta(cache_wheel_on_google_drive=False, serialization=False, mirror=0)
import pyrosetta; pyrosetta.init()
from pyrosetta import *
from pyrosetta.teaching import *

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r2 2026.29+releasequarter

In [15]:
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyrosetta
from pyrosetta import rosetta

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

In [16]:
pdb_path = None
original_pdb_name = None

upload_button = widgets.Button(
    description="Upload PDB",
    button_style="info",
    icon="upload"
)

upload_output = widgets.Output()


def upload_pdb(_):

    global pdb_path
    global original_pdb_name

    with upload_output:

        clear_output()

        uploaded = files.upload()

        pdbs = [
            name
            for name in uploaded.keys()
            if name.lower().endswith(".pdb")
        ]

        if not pdbs:
            print("No PDB file was uploaded.")
            return

        original_pdb_name = pdbs[0]

        # Use a simple internal filename
        pdb_path = "/content/input_complex.pdb"

        shutil.copy2(
            original_pdb_name,
            pdb_path
        )

        print(
            f"Uploaded: {original_pdb_name}"
        )

        print(
            f"Internal path: {pdb_path}"
        )


upload_button.on_click(upload_pdb)

display(
    upload_button,
    upload_output
)

Saving threaded_model (1).relaxed.pdb to threaded_model (1).relaxed.pdb
Uploaded: threaded_model (1).relaxed.pdb
Internal path: /content/input_complex.pdb


In [22]:
init_options = "-mute all"

pyrosetta.init(
    init_options
)

print("PyRosetta initialized.")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r2 2026.29+releasequarter

In [23]:
scorefxn = pyrosetta.create_score_function(
    "ref2015"
)

print("Score function: ref2015")

Score function: ref2015


In [24]:
ligand_name_widget = widgets.Text(
    value="LIG",
    description="Ligand name:",
    layout=widgets.Layout(width="300px")
)

display(
    ligand_name_widget
)

Text(value='LIG', description='Ligand name:', layout=Layout(width='300px'))

In [25]:
pose = pyrosetta.pose_from_pdb(
    pdb_path
)

print(
    f"Pose contains {pose.total_residue()} residues."
)

print()
print("Non-polymer residues:")
print("-" * 50)

for i in range(
    1,
    pose.total_residue() + 1
):

    residue = pose.residue(i)

    if not residue.is_polymer():

        print(
            f"Pose {i:4d} | "
            f"{residue.name3():<6} | "
            f"PDB: {pose.pdb_info().pose2pdb(i)}"
        )

Pose contains 182 residues.

Non-polymer residues:
--------------------------------------------------
Pose  181 |  MG    | PDB: 616 B 
Pose  182 | ADP    | PDB: 616 C 


In [26]:
def find_ligand_residues(
    pose,
    ligand_name
):

    ligand_name = ligand_name.upper()

    ligand_residues = []

    for i in range(
        1,
        pose.total_residue() + 1
    ):

        residue = pose.residue(i)

        if residue.name3().strip().upper() == ligand_name:

            ligand_residues.append(i)

    return ligand_residues

In [27]:
ligand_name = (
    ligand_name_widget.value.strip().upper()
)

ligand_residues = find_ligand_residues(
    pose,
    ligand_name
)

print(
    "Ligand:",
    ligand_name
)

print(
    "Pose residue(s):",
    ligand_residues
)

Ligand: LIG
Pose residue(s): []


In [28]:
def get_protein_residues(
    pose
):

    protein_residues = []

    for i in range(
        1,
        pose.total_residue() + 1
    ):

        if pose.residue(i).is_protein():

            protein_residues.append(i)

    return protein_residues

In [29]:
def separate_ligand(
    pose,
    ligand_residues,
    distance=500.0
):

    separated = pose.clone()

    translation = (
        rosetta.numeric.xyzVector_double_t(
            distance,
            0.0,
            0.0
        )
    )

    for resnum in ligand_residues:

        residue = separated.residue(
            resnum
        )

        for atomno in range(
            1,
            residue.natoms() + 1
        ):

            atom_id = rosetta.core.id.AtomID(
                atomno,
                resnum
            )

            old_xyz = separated.xyz(
                atom_id
            )

            new_xyz = (
                old_xyz
                + translation
            )

            separated.set_xyz(
                atom_id,
                new_xyz
            )

    return separated

In [31]:
def get_score_terms(
    pose,
    scorefxn
):

    # Score the pose first so Energies is up to date
    total_score = float(
        scorefxn(pose)
    )

    energies = pose.energies()

    energy_map = (
        energies.total_energies()
    )

    terms = {}

    # Get all non-zero weighted terms
    score_types = (
        scorefxn
        .get_nonzero_weighted_scoretypes()
    )

    for score_type in score_types:

        name = (
            rosetta.core.scoring
            .name_from_score_type(
                score_type
            )
        )

        raw_value = float(
            energy_map[score_type]
        )

        weight = float(
            scorefxn.get_weight(
                score_type
            )
        )

        weighted_value = (
            raw_value * weight
        )

        terms[name] = {
            "raw": raw_value,
            "weight": weight,
            "weighted": weighted_value
        }

    return total_score, terms

In [33]:
def calculate_ligand_interaction(
    pose,
    ligand_residues,
    scorefxn
):

    # --------------------------------------------------
    # Complex state
    # --------------------------------------------------

    complex_pose = pose.clone()

    complex_score, complex_terms = (
        get_score_terms(
            complex_pose,
            scorefxn
        )
    )

    # --------------------------------------------------
    # Separated state
    # --------------------------------------------------

    separated_pose = separate_ligand(
        pose,
        ligand_residues,
        distance=500.0
    )

    separated_score, separated_terms = (
        get_score_terms(
            separated_pose,
            scorefxn
        )
    )

    # --------------------------------------------------
    # Total interaction energy
    # --------------------------------------------------

    interaction_energy = (
        complex_score
        - separated_score
    )

    # --------------------------------------------------
    # Term-by-term interaction energy
    # --------------------------------------------------

    term_rows = []

    all_terms = sorted(
        set(complex_terms.keys())
        |
        set(separated_terms.keys())
    )

    for term in all_terms:

        bound_raw = (
            complex_terms
            .get(term, {})
            .get("raw", 0.0)
        )

        separated_raw = (
            separated_terms
            .get(term, {})
            .get("raw", 0.0)
        )

        weight = (
            complex_terms
            .get(term, {})
            .get(
                "weight",
                separated_terms
                .get(term, {})
                .get("weight", 0.0)
            )
        )

        weighted_bound = (
            bound_raw * weight
        )

        weighted_separated = (
            separated_raw * weight
        )

        interaction = (
            weighted_bound
            - weighted_separated
        )

        term_rows.append({

            "ScoreTerm": term,

            "Weight": weight,

            "Complex_raw": bound_raw,

            "Separated_raw":
                separated_raw,

            "Complex_weighted":
                weighted_bound,

            "Separated_weighted":
                weighted_separated,

            "Interaction_REU":
                interaction
        })

    term_df = pd.DataFrame(
        term_rows
    )

    # Sort by magnitude of contribution
    term_df["Abs_interaction"] = (
        term_df["Interaction_REU"]
        .abs()
    )

    term_df = (
        term_df
        .sort_values(
            "Abs_interaction",
            ascending=False
        )
        .drop(
            columns=["Abs_interaction"]
        )
        .reset_index(drop=True)
    )

    return {
        "complex_score":
            complex_score,

        "separated_score":
            separated_score,

        "interaction_energy":
            interaction_energy,

        "complex_pose":
            complex_pose,

        "separated_pose":
            separated_pose,

        "term_df":
            term_df
    }

In [32]:
run_button = widgets.Button(
    description="Analyze protein-ligand interaction",
    button_style="success",
    icon="play"
)

run_output = widgets.Output()

analysis_result = None


def run_analysis(_):

    global analysis_result

    with run_output:

        clear_output()

        try:

            # ----------------------------------------------
            # Load PDB
            # ----------------------------------------------

            if pdb_path is None:

                raise ValueError(
                    "Please upload a PDB first."
                )

            pose = pyrosetta.pose_from_pdb(
                pdb_path
            )

            ligand_name = (
                ligand_name_widget
                .value
                .strip()
                .upper()
            )

            # ----------------------------------------------
            # Find ligand
            # ----------------------------------------------

            ligand_residues = (
                find_ligand_residues(
                    pose,
                    ligand_name
                )
            )

            if not ligand_residues:

                raise ValueError(
                    f"Ligand '{ligand_name}' "
                    "was not found in the PDB."
                )

            print("=" * 70)
            print("PROTEIN-LIGAND ENERGY ANALYSIS")
            print("=" * 70)

            print(
                f"PDB: {original_pdb_name}"
            )

            print(
                f"Ligand: {ligand_name}"
            )

            print(
                f"Ligand pose residue(s): "
                f"{ligand_residues}"
            )

            print()

            # ----------------------------------------------
            # Calculate
            # ----------------------------------------------

            analysis_result = (
                calculate_ligand_interaction(
                    pose,
                    ligand_residues,
                    scorefxn
                )
            )

            # ----------------------------------------------
            # Scores
            # ----------------------------------------------

            complex_score = (
                analysis_result[
                    "complex_score"
                ]
            )

            separated_score = (
                analysis_result[
                    "separated_score"
                ]
            )

            interaction_energy = (
                analysis_result[
                    "interaction_energy"
                ]
            )

            print(
                f"Complex score:     "
                f"{complex_score:.3f} REU"
            )

            print(
                f"Separated score:   "
                f"{separated_score:.3f} REU"
            )

            print(
                f"Interaction energy: "
                f"{interaction_energy:.3f} REU"
            )

            print()

            # ----------------------------------------------
            # Term table
            # ----------------------------------------------

            term_df = (
                analysis_result[
                    "term_df"
                ]
            )

            print(
                "Energy-term decomposition:"
            )

            display(
                term_df
            )

            # ----------------------------------------------
            # Plot contributions
            # ----------------------------------------------

            plot_df = (
                term_df
                .sort_values(
                    "Interaction_REU"
                )
            )

            plt.figure(
                figsize=(10, 7)
            )

            plt.barh(
                plot_df["ScoreTerm"],
                plot_df[
                    "Interaction_REU"
                ]
            )

            plt.axvline(
                0,
                linewidth=1
            )

            plt.xlabel(
                "Protein-ligand interaction energy (REU)"
            )

            plt.ylabel(
                "Score term"
            )

            plt.title(
                "Protein-ligand interaction energy decomposition"
            )

            plt.tight_layout()

            plt.show()

        except Exception as e:

            print(
                "ERROR:",
                type(e).__name__,
                e
            )


run_button.on_click(
    run_analysis
)

display(
    run_button,
    run_output
)

Button(button_style='success', description='Analyze protein-ligand interaction', icon='play', style=ButtonStyl…

Output()

In [34]:
save_button = widgets.Button(
    description="Save results",
    button_style="warning",
    icon="save"
)

save_output = widgets.Output()


def save_results(_):

    with save_output:

        clear_output()

        if analysis_result is None:

            print(
                "Run the analysis first."
            )

            return

        term_df = (
            analysis_result[
                "term_df"
            ]
        )

        # ------------------------------------------
        # Summary
        # ------------------------------------------

        summary_df = pd.DataFrame({

            "PDB": [
                original_pdb_name
            ],

            "Ligand": [
                ligand_name_widget.value
            ],

            "Complex_score_REU": [
                analysis_result[
                    "complex_score"
                ]
            ],

            "Separated_score_REU": [
                analysis_result[
                    "separated_score"
                ]
            ],

            "Interaction_energy_REU": [
                analysis_result[
                    "interaction_energy"
                ]
            ]
        })

        summary_df.to_csv(
            "/content/"
            "protein_ligand_summary.csv",
            index=False
        )

        term_df.to_csv(
            "/content/"
            "protein_ligand_energy_terms.csv",
            index=False
        )

        print(
            "Saved:"
        )

        print(
            "  protein_ligand_summary.csv"
        )

        print(
            "  protein_ligand_energy_terms.csv"
        )


save_button.on_click(
    save_results
)

display(
    save_button,
    save_output
)

Button(button_style='warning', description='Save results', icon='save', style=ButtonStyle())

Output()

In [ ]:
download_button = widgets.Button(
    description="Download results",
    button_style="info",
    icon="download"
)

download_output = widgets.Output()


def download_results(_):

    with download_output:

        clear_output()

        summary_path = (
            "/content/"
            "protein_ligand_summary.csv"
        )

        terms_path = (
            "/content/"
            "protein_ligand_energy_terms.csv"
        )

        if not os.path.exists(
            summary_path
        ):

            print(
                "Run and save the analysis first."
            )

            return

        files.download(
            summary_path
        )

        files.download(
            terms_path
        )


download_button.on_click(
    download_results
)

display(
    download_button,
    download_output
)